## RQ5. What popular food catetgory in NOVA 3 and NOVA 4 contribute to health risks based on the ingredients when consumed frequently?

In [1]:
import pandas as pd

df = pd.read_csv("branded_food_cleaned_classified.csv")
df.head()

,fdc_id,brand_owner,branded_food_category,ingredients,market_country,nova_category
0,1105904,Richardson Oilseed Products (Us) Limited,Oils Edible,Vegetable Oil,United States,NOVA 2 - Culinary Ingredient
1,1105905,Campbell Soup Company,Herbs/Spices/Extracts,"INGREDIENTS: BEEF STOCK, CONTAINS LESS THAN 2%...",United States,NOVA 1 - Unprocessed/Minimally Processed
2,1105906,Campbell Soup Company,Prepared Soups,"INGREDIENTS: CLAM STOCK, POTATOES, CLAMS, CREA...",United States,NOVA 4 - Ultra-Processed
3,1105907,Campbell Soup Company,Prepared Soups,"INGREDIENTS: WATER, CREAM, BROCCOLI, CELERY, V...",United States,NOVA 4 - Ultra-Processed
4,1105908,Campbell Soup Company,Herbs/Spices/Extracts,"INGREDIENTS: CHICKEN STOCK, CONTAINS LESS THAN...",United States,NOVA 1 - Unprocessed/Minimally Processed


In [2]:
# NOVA 3 defining ingredients
nova3_ingredients = {
    "Salt": ["SALT", "SEA SALT", "SODIUM"],
    "Sugar": ["SUGAR", "CANE SUGAR", "BROWN SUGAR"],
    "Oil": ["OIL", "OLIVE OIL", "VEGETABLE OIL", "CANOLA OIL", "SUNFLOWER OIL"],
    "Vinegar": ["VINEGAR"],
    "Butter": ["BUTTER"],
    "Honey": ["HONEY"],
}

# NOVA 3 ingredient prevalence by NOVA tier
df_upf = df[df["nova_category"].isin(
    ["NOVA 3 - Processed", "NOVA 4 - Ultra-Processed"]
)].copy()

def has_marker(text, keywords):
    if pd.isna(text):
        return False
    text = str(text).upper()
    return any(kw in text for kw in keywords)
    
nova3_results = []
for category, keywords in nova3_ingredients.items():
    for tier in ["NOVA 3 - Processed", "NOVA 4 - Ultra-Processed"]:
        tier_df = df_upf[df_upf["nova_category"] == tier]
        count = tier_df["ingredients"].apply(lambda x: has_marker(x, keywords)).sum()
        pct = round(count / len(tier_df) * 100, 2)
        nova3_results.append({
            "ingredient": category,
            "nova_tier": tier,
            "product_count": count,
            "pct_of_tier": pct
        })

nova3_df = pd.DataFrame(nova3_results)
pivot3 = nova3_df.pivot(index="ingredient", columns="nova_tier", values="pct_of_tier")
pivot3.columns = ["NOVA 3 %", "NOVA 4 %"]
pivot3 = pivot3.sort_values("NOVA 3 %", ascending=False)
pivot3

,NOVA 3 %,NOVA 4 %
ingredient,,
Salt,73.22,78.70
Sugar,37.02,66.41
Oil,28.11,57.29
Vinegar,10.09,11.28
Butter,4.61,23.35
Honey,3.76,6.10


In [3]:
pivot3.to_csv("nova3_ingredient_comparison.csv")
print("Exported NOVA 3 comparison")

Exported NOVA 3 comparison


In [4]:
# Top 20 food categories containing NOVA 3 ingredients
all_nova3_kw = [kw for keywords in nova3_ingredients.values() for kw in keywords]

df_upf["has_nova3_ingredient"] = df_upf["ingredients"].apply(
    lambda x: has_marker(x, all_nova3_kw)
)

cat_nova3 = (df_upf.groupby("branded_food_category")
             .agg(total=("has_nova3_ingredient", "size"),
                  with_ingredient=("has_nova3_ingredient", "sum"))
             .assign(pct=lambda x: (x["with_ingredient"] / x["total"] * 100).round(2))
             .query("total >= 100")
             .sort_values("pct", ascending=False))

# Export all, display top 20
cat_nova3.to_csv("nova3_ingredients_by_category.csv")
print(f"Exported {len(cat_nova3)} food categories")
cat_nova3.head(20)

Exported 106 food categories


,total,with_ingredient,pct
branded_food_category,,,
Bread,4479,4479,100.00
Biscuits/Cookies (Shelf Stable),1048,1048,100.00
Bread/Bakery Products Variety Packs,284,284,100.00
Bread (Frozen),215,215,100.00
Cakes Sweet (Frozen),156,156,100.00
Butter/Butter Substitutes,227,227,100.00
Chips/Crisps/Snack Mixes Natural/Extruded (Shelf Stable),322,322,100.00
Cream/Cream Substitutes,228,228,100.00
Frozen Bread & Dough,4963,4963,100.00


In [5]:
# NOVA 4 marker categories (Monteiro et al., 2019):
nova4_markers = {
    "Modified sugars & syrups": [
        "CORN SYRUP", "HIGH FRUCTOSE", "DEXTROSE", "MALTODEXTRIN",
        "GLUCOSE SYRUP", "INVERT SUGAR", "CORN SYRUP SOLIDS", "RICE SYRUP",
    ],
    "Hydrogenated oils": [
        "HYDROGENATED", "PARTIALLY HYDROGENATED", "INTERESTERIFIED",
    ],
    "Protein isolates & hydrolysates": [
        "PROTEIN ISOLATE", "PROTEIN CONCENTRATE", "HYDROLYZED",
        "CASEIN", "CASEINATE",
    ],
    "Modified starches": [
        "MODIFIED STARCH", "MODIFIED CORN", "MODIFIED FOOD STARCH",
        "MODIFIED TAPIOCA",
    ],
    "Flavours": [
        "FLAVOR", "FLAVORING", "FLAVOUR", "VANILLIN",
    ],
    "Flavour enhancers": [
        "MONOSODIUM GLUTAMATE", "MSG", "YEAST EXTRACT", "AUTOLYZED YEAST",
        "DISODIUM INOSINATE", "DISODIUM GUANYLATE",
    ],
    "Colours": [
        "RED 40", "RED 3", "YELLOW 5", "YELLOW 6", "BLUE 1", "BLUE 2",
        "GREEN 3", "CARAMEL COLOR", "TITANIUM DIOXIDE", "ANNATTO",
        "COLOR ADDED", "ARTIFICIAL COLOR",
    ],
    "Emulsifiers": [
        "LECITHIN", "DIGLYCERIDES", "POLYSORBATE", "CARRAGEENAN",
        "DATEM", "SORBITAN", "SODIUM STEAROYL LACTYLATE",
    ],
    "Sweeteners (non-sugar)": [
        "ASPARTAME", "SUCRALOSE", "ACESULFAME", "SACCHARIN", "STEVIA",
        "ERYTHRITOL", "SORBITOL", "MANNITOL", "XYLITOL", "NEOTAME",
    ],
    "Thickeners & gums": [
        "XANTHAN GUM", "GUAR GUM", "CELLULOSE GUM", "CELLULOSE",
        "LOCUST BEAN GUM", "GELLAN GUM", "PECTIN", "CARBOXYMETHYL",
    ],
    "Preservatives": [
        "SODIUM BENZOATE", "POTASSIUM SORBATE", "BHT", "BHA", "TBHQ",
        "CALCIUM PROPIONATE", "SODIUM NITRITE", "SODIUM NITRATE",
        "TOCOPHEROLS", "SULFITE", "BISULFITE",
    ],
    "Anti-caking & other agents": [
        "SILICON DIOXIDE", "CALCIUM SILICATE", "SHELLAC", "CARNAUBA WAX",
        "PROPYLENE GLYCOL", "POLYDEXTROSE",
    ],
}

In [6]:
# Top 20 food categories containing ANY NOVA 4 marker
all_keywords = [kw for keywords in nova4_markers.values() for kw in keywords]

df_upf["has_nova4_marker"] = df_upf["ingredients"].apply(
    lambda x: has_marker(x, all_keywords)
)


cat_all = (df_upf.groupby("branded_food_category")
           .agg(total=("has_nova4_marker", "size"),
                with_marker=("has_nova4_marker", "sum"))
           .assign(pct=lambda x: (x["with_marker"] / x["total"] * 100).round(2))
           .query("total >= 100")
           .sort_values("pct", ascending=False))

print("\n=== Top 20 Food Categories by % Containing NOVA 4 Markers ===")
cat_all.head(20)


=== Top 20 Food Categories by % Containing NOVA 4 Markers ===


,total,with_marker,pct
branded_food_category,,,
Bread/Bakery Products Variety Packs,284,284,100.00
Cakes Sweet (Frozen),156,156,100.00
Pies/Pastries Sweet (Shelf Stable),216,216,100.00
Biscuits/Cookies (Shelf Stable),1048,1047,99.90
Sandwiches/Filled Rolls/Wraps,1948,1946,99.90
Chewing Gum & Mints,6492,6457,99.46
Cream/Cream Substitutes,228,226,99.12
Confectionery Products,2058,2035,98.88
Butter/Butter Substitutes,227,224,98.68


In [7]:
# Saving the file to csv
cat_all.to_csv("nova4_markers_by_category.csv")
print(f"Exported {len(cat_all)} food categories to nova4_markers_by_category.csv")

Exported 106 food categories to nova4_markers_by_category.csv


In [8]:
# Filter to NOVA 3 + 4 
df_upf = df[df["nova_category"].isin(
    ["NOVA 3 - Processed", "NOVA 4 - Ultra-Processed"]
)].copy()

# Check each product's ingredients for marker keywords
def has_marker(text, keywords):
    if pd.isna(text):
        return False
    text = str(text).upper()
    return any(kw in text for kw in keywords)


# 1. Marker prevalence by category, split by NOVA tier
results = []
for category, keywords in nova4_markers.items():
    for tier in ["NOVA 3 - Processed", "NOVA 4 - Ultra-Processed"]:
        tier_df = df_upf[df_upf["nova_category"] == tier]
        count = tier_df["ingredients"].apply(lambda x: has_marker(x, keywords)).sum()
        pct = round(count / len(tier_df) * 100, 2)
        results.append({
            "marker_category": category,
            "nova_tier": tier,
            "product_count": count,
            "pct_of_tier": pct
        })

marker_results = pd.DataFrame(results)

pivot = marker_results.pivot(index="marker_category", columns="nova_tier", values="pct_of_tier")
pivot.columns = ["NOVA 3 %", "NOVA 4 %"]
pivot["Difference"] = (pivot["NOVA 4 %"] - pivot["NOVA 3 %"]).round(2)
pivot = pivot.sort_values("NOVA 4 %", ascending=False)
pivot

,NOVA 3 %,NOVA 4 %,Difference
marker_category,,,
Flavours,32.08,63.80,31.72
Modified sugars & syrups,13.69,49.04,35.35
Emulsifiers,22.79,35.03,12.24
Colours,15.66,30.32,14.66
Thickeners & gums,25.79,24.06,-1.73
Preservatives,17.24,22.54,5.30
Modified starches,9.82,16.24,6.42
Protein isolates & hydrolysates,7.09,11.54,4.45
Flavour enhancers,1.65,11.04,9.39


In [9]:
# Saving the result difference
pivot.to_csv("nova4_marker_comparison.csv")
print("Exported to nova4_marker_comparison.csv")

Exported to nova4_marker_comparison.csv


The results show a clear distinction between NOVA 3 and NOVA 4 products. NOVA 4 markers (in cell 3 from the code) such as flavours (63.80%), modified sugars and syrups (49.04%), emulsifiers (35.03%), and colours (30.32%) are significantly more prevalent in NOVA 4 compared to NOVA 3. Among popular food categories, nearly all products in categories like candy (98.61%), cakes and snack cakes (97.82%), ice cream and frozen yogurt (97.11%), and cookies (99.90%) contain at least one NOVA 4 marker. These findings confirm that ultra-processed foods are defined not just by excess salt and sugar, but by the industrial additives layered on top - the same additives that peer-reviewed research links to obesity, cardiovascular disease, and metabolic syndrome.